# Rory's Travel Club — Dashboard (dev notebook)

This mirrors `app.py`, split into cells so you can poke at the data-loading and filtering logic while developing.

**Heads up:** Streamlit widgets (`st.button`, `st.multiselect`, etc.) need a real Streamlit server to be interactive — running this notebook cell by cell will execute them but they won't respond to clicks here, and some may print a harmless "missing ScriptRunContext" warning. Use this notebook to iterate on the logic, then run the real thing with:

```bash
streamlit run app.py
```

In [1]:
import os
from datetime import date

import pandas as pd
import streamlit as st

from scraper import scrape_all, LOCATIONS

CSV_PATH = "deals.csv"

ModuleNotFoundError: No module named 'scraper'

## Load data
Same loader `app.py` uses — reads `deals.csv` if present.

In [ ]:
def load_from_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["expiry_date"] = pd.to_datetime(df["expiry_date"], errors="coerce")
    return df

if os.path.exists(CSV_PATH):
    df = load_from_csv(CSV_PATH)
else:
    df = scrape_all(LOCATIONS)
    df.to_csv(CSV_PATH, index=False)

df.head()

## Filtering logic
Pulled out as a plain function (no Streamlit calls) so you can test it directly on `df` here, independent of the UI.

In [ ]:
def filter_deals(df, search="", counties=None, provinces=None, sort_by="Soonest to expire"):
    filtered = df.copy()

    if search:
        s = search.lower()
        mask = (
            filtered["hotel"].str.lower().str.contains(s, na=False)
            | filtered["county"].str.lower().str.contains(s, na=False)
            | filtered["description"].str.lower().str.contains(s, na=False)
        )
        filtered = filtered[mask]

    if counties:
        filtered = filtered[filtered["county"].isin(counties)]

    if provinces:
        filtered = filtered[filtered["province"].isin(provinces)]

    filtered["days_left"] = (filtered["expiry_date"] - pd.Timestamp(date.today())).dt.days

    if sort_by == "Soonest to expire":
        filtered = filtered.sort_values("days_left", na_position="last")
    elif sort_by == "Lowest price first":
        filtered = filtered.sort_values("min_price", na_position="last")
    else:
        filtered = filtered.sort_values("hotel")

    return filtered

# Try it out, e.g.:
filter_deals(df, search="dublin", sort_by="Lowest price first")

## Full Streamlit UI
This is the same code as `app.py`. Running this cell inside Jupyter will execute it top to bottom but won't give you the interactive browser dashboard — for that, run `streamlit run app.py` from a terminal in this folder.

In [ ]:
st.set_page_config(page_title="Rory's Travel Club — Deals Board", page_icon="🧳", layout="wide")

# --- Sidebar: data controls -------------------------------------------------
st.sidebar.header("Data")
if st.sidebar.button("🔄 Scrape live now"):
    with st.spinner("Scraping rorystravelclub.com..."):
        fresh = scrape_all(LOCATIONS)
        fresh.to_csv(CSV_PATH, index=False)
        df = fresh
    st.sidebar.success(f"Pulled {len(fresh)} deals")

if df.empty:
    st.warning(
        "No data yet. Run the scraper cells above first, "
        "or click **Scrape live now** in the sidebar."
    )
    st.stop()

# --- Sidebar: filters --------------------------------------------------------
st.sidebar.header("Filters")
search = st.sidebar.text_input("Search hotel, county, or description")

counties = sorted(df["county"].dropna().unique())
county_choice = st.sidebar.multiselect("County", counties, default=[])

provinces = sorted(df["province"].dropna().unique())
province_choice = st.sidebar.multiselect("Province", provinces, default=[])

sort_choice = st.sidebar.selectbox(
    "Sort by", ["Soonest to expire", "Lowest price first", "Hotel name (A-Z)"]
)

filtered = filter_deals(df, search, county_choice, province_choice, sort_choice)

# --- Header / stats ----------------------------------------------------------
st.title("🧳 Rory's Travel Club — Deals Board")
st.caption("Every live hotel offer, searchable and filterable.")

c1, c2, c3 = st.columns(3)
c1.metric("Live deals", len(df))
c2.metric("Counties", df["county"].nunique())
soonest = df["expiry_date"].dropna()
if not soonest.empty:
    days_to_soonest = (soonest.min() - pd.Timestamp(date.today())).days
    c3.metric("Days to nearest expiry", max(days_to_soonest, 0))
else:
    c3.metric("Days to nearest expiry", "—")

st.divider()

# --- Deal cards ---------------------------------------------------------------
if filtered.empty:
    st.info("No deals match those filters.")
else:
    cols = st.columns(3)
    for i, (_, deal) in enumerate(filtered.iterrows()):
        col = cols[i % 3]
        with col:
            with st.container(border=True):
                if pd.notna(deal.get("image_url")):
                    st.image(deal["image_url"], use_container_width=True)

                st.markdown(f"**{deal['hotel']}**")
                st.caption(f"📍 {str(deal['county']).title()} · {deal.get('province', '')}")
                st.write(deal["description"])

                price_col, expiry_col = st.columns(2)
                with price_col:
                    if pd.notna(deal.get("min_price")):
                        st.markdown(f"💶 from {deal.get('currency', '')}{deal['min_price']:.0f}")
                with expiry_col:
                    days_left = deal.get("days_left")
                    if pd.notna(days_left):
                        if days_left < 0:
                            st.markdown(":red[Expired]")
                        elif days_left <= 21:
                            st.markdown(f":red[⏰ {int(days_left)}d left]")
                        else:
                            st.markdown(f"⏰ {int(days_left)}d left")
                    else:
                        st.markdown(f"⏰ {deal['expiry']}")

                if pd.notna(deal.get("offer_url")):
                    st.link_button("View deal ↗", deal["offer_url"], use_container_width=True)

st.divider()
st.caption(
    "Data columns match the scraper's DataFrame: hotel, county, description, "
    "expiry, offer_url, image_url (plus derived province/price/expiry_date). "
    "Add more province slugs to LOCATIONS in scraper.py as you find them."
)